In [ ]:
import os
import email
import pandas as pd

def extract_email_text(file_path):
    """Extract text from an .eml file using the email library.
       If the email is multipart, it returns the concatenated text/plain parts.
    """
    text_content = ""
    try:
        with open(file_path, 'r', encoding='utf-8', errors='replace') as f:
            msg = email.message_from_file(f)
    except Exception as e:
        print(f"Error reading {file_path}: {e}")
        return text_content

    # If the email is multipart, iterate over its parts.
    if msg.is_multipart():
        for part in msg.walk():
            # Look for text/plain parts that are not attachments.
            if part.get_content_type() == "text/plain" and not part.get('Content-Disposition'):
                try:
                    charset = part.get_content_charset() or 'utf-8'
                    part_text = part.get_payload(decode=True).decode(charset, errors='replace')
                    text_content += part_text + "\n"
                except Exception as e:
                    print(f"Error decoding part of {file_path}: {e}")
    else:
        # Single-part email.
        try:
            payload = msg.get_payload(decode=True)
            if payload:
                charset = msg.get_content_charset() or 'utf-8'
                text_content = payload.decode(charset, errors='replace')
            else:
                text_content = ""
        except Exception as e:
            print(f"Error decoding {file_path}: {e}")
    
    return text_content.strip()

# # Set the root directory containing your .eml files (including subfolders)
# root_folder = r"C:\Emails_Archiving\Michael Brewer Email_converted\1Admin.pst\1 Admin\Top of Personal Folders"

# Set the root directory containing your .eml files (including subfolders)
root_folder = r"C:\Emails_Archiving\Michael_Brewer_Email_Converted"


# List to collect email data
emails_data = []

# Walk through the folder structure recursively.
for dirpath, dirnames, filenames in os.walk(root_folder):
    for filename in filenames:
        if filename.lower().endswith(".eml"):
            file_path = os.path.join(dirpath, filename)
            email_text = extract_email_text(file_path)
            emails_data.append({
                "file_name": filename,
                "file_path": file_path,
                "email_text": email_text
            })

# Create a DataFrame from the collected data.
df_emails = pd.DataFrame(emails_data)


# Save the DataFrame to a CSV file.
output_csv = r"C:\Emails_Archiving\emails_parsed.csv"
df_emails.to_csv(output_csv, index=False)

print(f"Parsed {len(df_emails)} emails and saved to {output_csv}")

PermissionError: [Errno 13] Permission denied: 'C:\\Emails_Archiving\\emails_parsed.csv'

Parsed 44677 emails and saved to C:\Emails_Archiving\emails_parsed.csv
